In [2]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

drive_root = Path("/content/drive/MyDrive/vietnamese-summarization")

if not drive_root.is_dir():
    raise RuntimeError("Lỗi: Không tìm thấy thư mục dự án trên Drive.")

print("Đã kết nối Drive.")

Mounted at /content/drive
Đã kết nối Drive.


In [3]:
import os
import sys
import subprocess

repo_url = "https://github.com/MD1809/vietnamese-summarization.git"
repo_dir = Path("/content/vietnamese-summarization")
target_commit = "bd92efd65cd21ed2add88defd3a110e931f5d573"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", repo_url, str(repo_dir)],
        check=True,
    )

if not (repo_dir / ".git").is_dir():
    raise RuntimeError("Lỗi: Thư mục dự án không phải Git repository.")

remote_result = subprocess.run(
    ["git", "remote", "get-url", "origin"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if remote_result.stdout.strip() != repo_url:
    raise RuntimeError("Lỗi: Repository có origin khác với dự án.")

status_result = subprocess.run(
    ["git", "status", "--porcelain"],
    cwd=repo_dir,
    capture_output=True,
    text=True,
    check=True,
)

if status_result.stdout.strip() != "":
    raise RuntimeError(
        "Lỗi: Repository có thay đổi chưa lưu. "
        "Cần kiểm tra trước khi chuyển commit."
    )

subprocess.run(
    ["git", "fetch", "origin"],
    cwd=repo_dir,
    check=True,
)

subprocess.run(
    ["git", "checkout", "--detach", target_commit],
    cwd=repo_dir,
    check=True,
)

os.chdir(repo_dir)

if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

print("Commit đang sử dụng:", target_commit)
print("Thư mục làm việc:", Path.cwd())

Commit đang sử dụng: bd92efd65cd21ed2add88defd3a110e931f5d573
Thư mục làm việc: /content/vietnamese-summarization


In [4]:
%pip uninstall -y diffusers
%pip install "jedi==0.20.1"
%pip install -r /content/vietnamese-summarization/requirements.txt
%pip check

Found existing installation: diffusers 0.42.1
Uninstalling diffusers-0.42.1:
  Successfully uninstalled diffusers-0.42.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 12.8 MB/s  0:00:00
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu130
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 139.1 MB/s  0:00:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 168.7 MB/s  0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 221.9 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 58.0 MB/s  0:00:05
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 78.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 25.5 MB/s  0:00:08
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
from google.colab import files

uploaded = files.upload()

if "data.py" not in uploaded:
    raise RuntimeError("Lỗi: Chưa chọn đúng file data.py.")

data_module_path = repo_dir / "src" / "data.py"
data_module_path.write_bytes(uploaded["data.py"])

print("Đã thêm src/data.py vào runtime.")

Saving data.py to data.py
Đã thêm src/data.py vào runtime.


In [7]:
import json
from src.data import load_wikilingua, save_source_manifest

cache_dir = drive_root / "data" / "raw"
manifest_path = drive_root / "data" / "manifests" / "wikilingua_source.json"

dataset = load_wikilingua(cache_dir)

manifest = save_source_manifest(
    dataset,
    manifest_path,
    cache_dir,
)

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("Đã lưu manifest:", manifest_path)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

vietnamese/train-00000-of-00001.parquet:   0%|          | 0.00/33.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6616 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

{
  "created_at_utc": "2026-11-04T11:50:06.188502+00:00",
  "dataset_id": "esdurmus/wiki_lingua",
  "revision": "ea3db3510cbd34d0f8dc612419ae40e4732f3b40",
  "language": "vietnamese",
  "row_unit": "article; section pairs have not been extracted",
  "splits": {
    "train": {
      "article_row_count": 6616,
      "features": {
        "url": {
          "dtype": "string",
          "_type": "Value"
        },
        "article": {
          "section_name": {
            "feature": {
              "dtype": "string",
              "_type": "Value"
            },
            "_type": "List"
          },
          "document": {
            "feature": {
              "dtype": "string",
              "_type": "Value"
            },
            "_type": "List"
          },
          "summary": {
            "feature": {
              "dtype": "string",
              "_type": "Value"
            },
            "_type": "List"
          },
          "english_url": {
            "feature": {
   

In [8]:
files.download(str(manifest_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>